In [ ]:
mut = 3
pssm_sub = final_score_df[
    (final_score_df["mutations"] == mut) &
    (final_score_df["comparision_type"] == comp_type) &
    (final_score_df["score_type"] == "avg_score")
]

plm_sub = df_plot[
    (df_plot["model"] == "esm_35m") &
    (df_plot["num_mutations"] == mut)
]

In [ ]:
continuuous_activity_col = "activity"#pte_activity[3]

# ------- Overlayed ROC Curves for All Models: Binary Classification (Active/Inactive) -------

import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, auc
from scipy.stats import spearmanr

model_names = [col for col in normed_fitness.columns if col in color_map]
title_fontsize = 9
label_fontsize = 8
tick_fontsize = 7


log_activity = True



# Use binned_activity (should be 0: inactive, 1: active)
#y_true = df["binned_activity"].astype(float)
#y_true = df[continuuous_activity_col]
#y_true = (df[continuuous_activity_col] > np.median(df[continuuous_activity_col])).astype(int)
y_true = get_labels(d)
#y_true = (df[continuuous_activity_col] > 1).astype(int)

metric_name, metric_df = "Normed Fitness", normed_fitness

plt.figure(figsize=(8, 6))
roc_results = []
for col in model_names:
    # Select valid entries for this model
    selection = (~metric_df[col].isna()) & (~y_true.isna())
    if selection.sum() == 0:
        continue
    y_score = metric_df.loc[selection, col]
    y_true_sel = y_true.loc[selection]
    # Compute ROC curve and AUC
    try:
        fpr, tpr, thresholds = roc_curve(y_true_sel, y_score)
        roc_auc = auc(fpr, tpr)
    except Exception as e:
        continue
    label = f"{col} (AUC = {roc_auc:.3f})"
    plt.plot(fpr, tpr, color=color_map.get(col, "grey"), lw=2, label=label)
    roc_results.append((col, roc_auc))
# Plot random guess diagonal
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate", fontsize=label_fontsize)
plt.ylabel("True Positive Rate", fontsize=label_fontsize)
plt.title(f"Overlayed ROC Curves: {metric_name}", fontsize=title_fontsize+2)
plt.legend(loc="lower right", fontsize=8)
plt.tick_params(axis='both', labelsize=tick_fontsize)
plt.tight_layout()
plt.show()

# Print AUCs summary
print(f"\nOverlayed ROC Curve AUCs for {metric_name}:")
print(f"{'Model':<16} {'AUC ROC':>10}")
for col, auc_val in roc_results:
    print(f"{col:<16} {auc_val:10.3f}")

# ------------- Violin plots of fitness/normed_fitness for active vs inactive and show AUC ROC/Spearman as well -----------------

import seaborn as sns

labels = {0: 'Inactive', 1: 'Active'}
colors = {0: 'skyblue', 1: 'salmon'}

for metric_name, metric_df in [("Normed Fitness", normed_fitness)]:
    print(f"\n=== {metric_name} Violin ===")
    columns = [col for col in metric_df.columns if col in color_map]
    results_table = []
    fig, axes = plt.subplots(1, len(columns), figsize=(min(18.25, 2.9*len(columns)), 3.65), squeeze=False)
    for i, col in enumerate(columns):
        ax = axes[0, i]
        selection = (~metric_df[col].isna()) & (~y_true.isna())
        # Explicitly make Label a Categorical with order [Inactive, Active]
        data_plot = pd.DataFrame({
            metric_name: metric_df.loc[selection, col],
            'binned_activity': y_true.loc[selection],
        })
        data_plot['Label'] = data_plot['binned_activity'].map(labels)
        # Ensure categorical order
        data_plot['Label'] = pd.Categorical(data_plot['Label'], categories=[labels[0], labels[1]], ordered=True)
        sns.violinplot(
            data=data_plot,
            x='Label',
            y=metric_name,
            hue='Label',
            ax=ax,
            palette=[colors[0], colors[1]],
            cut=0,
            linewidth=1,
            inner="box",
            legend=False,
            order=[labels[0], labels[1]],  # enforce order
            hue_order=[labels[0], labels[1]],  # enforce order for hues
        )
        # Compute AUC ROC
        try:
            auc_roc = roc_auc_score(y_true, data_plot[metric_name])
        except Exception as e:
            auc_roc = float('nan')
        # Compute Spearman correlation vs activity for violin plot:
        act_selection = (~metric_df[col].isna()) & (~df[continuuous_activity_col].isna())
        try:
            spearman_corr, _ = spearmanr(df.loc[act_selection, continuuous_activity_col], metric_df.loc[act_selection, col])
        except Exception as e:
            spearman_corr = float('nan')
        results_table.append((col, auc_roc, spearman_corr))
        ax.set_xticklabels([labels[0], labels[1]], rotation=45, fontsize=tick_fontsize)
        ax.tick_params(axis='both', labelsize=tick_fontsize)
        if i == 0:
            ax.set_ylabel(metric_name, fontsize=label_fontsize)
        else:
            ax.set_ylabel("", fontsize=label_fontsize)
        ax.set_xlabel("", fontsize=label_fontsize)
        ax.set_title(f"{col}\nAUC={auc_roc:.3f}\nρ={spearman_corr:.3f}", fontsize=title_fontsize)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()
    print(f"{'Model':<16} {'AUC ROC':>10} {'Spearman ρ':>12}")
    for m, aucval, rho in results_table:
        print(f"{m:<16} {aucval:10.3f} {rho:12.3f}")



# ------------- Scatter plots for each model: activity vs fitness and normed fitness -----------------

import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

model_names = [col for col in normed_fitness.columns if col in color_map]
title_fontsize = 9
label_fontsize = 8
tick_fontsize = 7

for metric_name, metric_df in [("Normed Fitness", normed_fitness)]:
    print(f"\n=== {metric_name} ===")
    n_models = len(model_names)
    results_table = []
    fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 3), squeeze=False)
    for idx, col in enumerate(model_names):
        ax = axes[0, idx]
        selection = (~metric_df[col].isna()) & (~df[continuuous_activity_col].isna()) & (~y_true.isna())
        x = df.loc[selection, continuuous_activity_col]
        if log_activity:
            x = np.log(x)
        y = metric_df.loc[selection, col]
        y_bin = y_true.loc[selection]
        # Compute AUC ROC (with check for only one class present)
        try:
            auc_roc = roc_auc_score(y_bin, y)
        except Exception as e:
            auc_roc = float('nan')
        # Compute Spearman correlation
        try:
            spearman_corr, _ = spearmanr(x, y)
        except Exception as e:
            spearman_corr = float('nan')
        results_table.append((col, auc_roc, spearman_corr))
        ax.scatter(x, y, color=color_map.get(col, "grey"), s=10, alpha=0.7, edgecolors='none')
        ax.set_title(f"{col}\nAUC={auc_roc:.3f}\nρ={spearman_corr:.3f}", fontsize=title_fontsize)
        ax.set_xlabel("log(Activity)" if log_activity else "Activity", fontsize=label_fontsize)
        if idx == 0:
            ax.set_ylabel(metric_name, fontsize=label_fontsize)
        else:
            ax.set_ylabel("")
        ax.tick_params(axis='both', which='major', labelsize=tick_fontsize)
    fig.tight_layout()
    plt.show()

    # Print AUC ROC and Spearman correlation summary
    print(f"{'Model':<16} {'AUC ROC':>10} {'Spearman ρ':>12}")
    for m, auc, rho in results_table:
        print(f"{m:<16} {auc:10.3f} {rho:12.3f}")


In [ ]:
from collections import defaultdict

title_fontsize = 9
label_fontsize = 8
tick_fontsize = 7
legend_fontsize = 7

output_dir = "./figures/figure_1"
os.makedirs(output_dir, exist_ok=True)
fig_width = 18.25

mutation_counts = sorted(df["num_muts"].dropna().unique())
model_names = [col for col in normed_fitness.columns if col in color_map]
n_models = len(model_names)
n_mutations = len(mutation_counts)

for use_colormap in [False, True]:
    if use_colormap:
        fig_height = max(3.3, 0.4 * n_models)
    else:
        fig_height = max(3.7, 0.48 * n_models)
    fig, axes = plt.subplots(n_models, n_mutations, figsize=(fig_width, fig_height), squeeze=False)
    corrs = defaultdict(list)
    for m_idx, col_name in enumerate(model_names):
        for mut_idx, mut in enumerate(mutation_counts):
            selection = (~normed_fitness[col_name].isna()) & \
                        (~df["activity"].isna()) & \
                        (df["num_muts"] == mut)
            x = df.loc[selection, "activity"]
            y = normed_fitness.loc[selection, col_name]
            if len(x) == 0:
                rho = float('nan')
            else:
                rho, _ = spearmanr(y, x)
            if use_colormap:
                corrs[col_name].append(rho)
            plot_color = color_map.get(col_name, "gray") if use_colormap else "gray"
            axes[m_idx, mut_idx].scatter(x, y, color=plot_color, s=8, alpha=0.8, edgecolors='none')
            if m_idx == (n_models - 1):
                axes[m_idx, mut_idx].set_xlabel("log(Activity)", fontsize=label_fontsize)
            else:
                axes[m_idx, mut_idx].set_xlabel("")
                axes[m_idx, mut_idx].set_xticklabels([])
            if mut_idx == 0:
                axes[m_idx, mut_idx].set_ylabel("Fitness", fontsize=label_fontsize)
            else:
                axes[m_idx, mut_idx].set_ylabel("")
                axes[m_idx, mut_idx].set_yticklabels([])
            if m_idx == 0:
                axes[m_idx, mut_idx].set_title(f"{mut} mut\nRho={rho:.3f}", fontsize=title_fontsize)
            else:
                axes[m_idx, mut_idx].set_title(f"Rho={rho:.3f}", fontsize=title_fontsize)
            axes[m_idx, mut_idx].tick_params(axis='both', which='major', labelsize=tick_fontsize)
        axes[m_idx, 0].annotate(col_name, xy=(-0.54, 0.5), xycoords=axes[m_idx, 0].transAxes,
                        fontsize=label_fontsize, ha='right', va='center', rotation=90)
    plt.tight_layout()
    plt.show()
    #fname = "nmt_scatter_fitness_permutation" + ("_colored" if use_colormap else "") + ".svg"
    #fig.savefig(os.path.join(output_dir, fname), bbox_inches='tight')

In [ ]:
dataset_to_use = "nmt"

datasets_path = "./notebooks/%s/fitness_results/" % dataset_to_use
df = pd.read_csv(datasets[dataset_to_use])

normed_fitness_path = datasets_path + "normed_fitness_all.csv"
normed_fitness = pd.read_csv(normed_fitness_path)

fitness_path = datasets_path + "fitness_all.csv"
fitness = pd.read_csv(fitness_path)

In [ ]:
activity_values = df["activity"]
thresholds = np.linspace(activity_values.min(), activity_values.max(), 100)
counts_above = [(activity_values > t).sum() for t in thresholds]

plt.figure(figsize=(8, 5))
plt.plot(thresholds, counts_above, marker="o")
plt.xlabel("Activity Threshold")
plt.ylabel("Number of Rows Above Threshold")
plt.title("Number of Rows Above Activity Threshold")
plt.grid(True)
plt.show()

In [ ]:
print(normed_fitness.columns)

title_fontsize = 9
label_fontsize = 8
tick_fontsize = 7
legend_fontsize = 7

output_dir = "./figures/figure_1"
print(os.getcwd())
os.makedirs(output_dir, exist_ok=True)

fig_width = 8.25

all_cors = []

for use_colormap in [False, True]:
    if use_colormap:
        fig_height = 1.5
    else:
        fig_height = 1.7
    fig, axes = plt.subplots(1, len(normed_fitness.columns), figsize=(fig_width, fig_height), squeeze=False)
    for i, col_name in enumerate(normed_fitness.columns):
        valid = (~normed_fitness[col_name].isna()) & (~df["activity"].isna())
        normed_fit_col = normed_fitness.loc[valid, col_name]
        activity = df.loc[valid, "activity"]
        normed_corr_coef, normed_p_value = spearmanr(normed_fit_col, np.log(activity))

        if use_colormap:
            all_cors.append(normed_corr_coef)

        plot_color = color_map.get(col_name, "gray") if use_colormap else "gray"
        axes[0, i].scatter(np.log(activity), normed_fit_col, color=plot_color, s=8, alpha=0.8, edgecolors='none')
        axes[0, i].set_xlabel("log(Activity)", fontsize=label_fontsize)
        if i == 0:
            axes[0, i].set_ylabel("Fitness", fontsize=label_fontsize)
        if not use_colormap:
            axes[0, i].set_title(f"{col_name}\nRho={normed_corr_coef:.3f}", fontsize=title_fontsize)
        else:
            axes[0, i].set_title(f"Rho={normed_corr_coef:.3f}", fontsize=title_fontsize)
        axes[0, i].tick_params(axis='both', which='major', labelsize=tick_fontsize)
    # Use less tight spacing for use_colormap == True, more relaxed (wider) for False
    plt.tight_layout()
    plt.show()
    if use_colormap:
        fig.savefig(os.path.join(output_dir, "nmt_scatter_fitness_colored.svg"), bbox_inches='tight')
    else:
        fig.savefig(os.path.join(output_dir, "nmt_scatter_fitness.svg"), bbox_inches='tight')

In [ ]:


thresholds = [10, 100, 200, 500]
print(fitness.columns)

fig, axes = plt.subplots(2, len(fitness.columns), figsize=(15, 6), squeeze=False)

for i, col_name in enumerate(fitness.columns):
    valid = (~fitness[col_name].isna()) & (~df["activity"].isna())
    fit_col = fitness.loc[valid, col_name]
    activity = df.loc[valid, "activity"]
    aucs = []
    for t in thresholds:
        y_true = (activity > t).astype(int)
        if np.unique(y_true).size == 1:
            aucs.append(np.nan)
        else:
            aucs.append(roc_auc_score(y_true, fit_col))
    axes[0, i].plot(thresholds, aucs, marker="o")
    axes[0, i].set_xlabel("Activity Threshold")
    axes[0, i].set_ylabel("ROC AUC")
    axes[0, i].set_title(f"{col_name}\nAUC@thresholds")
    axes[0, i].set_ylim(0.3, 0.95)

    normed_fit_col = normed_fitness.loc[valid, col_name]
    normed_aucs = []

    for t in thresholds:
        y_true = (activity > t).astype(int)
        if np.unique(y_true).size == 1:
            normed_aucs.append(np.nan)
        else:
            normed_aucs.append(roc_auc_score(y_true, normed_fit_col))
    axes[1, i].plot(thresholds, normed_aucs, marker="o", color="red")
    axes[1, i].set_xlabel("Activity Threshold")
    axes[1, i].set_ylabel("ROC AUC")
    axes[1, i].set_title(f"Normed {col_name}\nAUC@thresholds")
    axes[1, i].set_ylim(0.3, 0.95)

plt.tight_layout()
plt.show()


In [ ]:
title_fontsize = 9
label_fontsize = 8
tick_fontsize = 7
legend_fontsize = 7

thresholds = [2, 3, 4, 5, 6]
print(fitness.columns)

nrows, ncols = 1, len(thresholds)
fig, axes = plt.subplots(nrows, ncols, figsize=(8.25, 2), squeeze=False)

raw_aucs = {col_name: [] for col_name in fitness.columns}
normed_aucs = {col_name: [] for col_name in fitness.columns}

for thresh_idx, t in enumerate(thresholds):
    for col_idx, col_name in enumerate(fitness.columns):
        valid = (~fitness[col_name].isna()) & (~df["activity"].isna())
        fit_col = fitness.loc[valid, col_name]
        normed_fit_col = normed_fitness.loc[valid, col_name]
        activity = df.loc[valid, "activity"]
        y_true = (np.log(activity) > (t)).astype(int)
        color = color_map.get(col_name, None)
        if np.unique(y_true).size == 2:
            fpr, tpr, _ = roc_curve(y_true, fit_col)
            roc_auc = auc(fpr, tpr)
            raw_aucs[col_name].append(roc_auc)
        else:
            raw_aucs[col_name].append(np.nan)
        if np.unique(y_true).size == 2:
            fpr_n, tpr_n, _ = roc_curve(y_true, normed_fit_col)
            roc_auc_n = auc(fpr_n, tpr_n)
            axes[0, thresh_idx].plot(fpr_n, tpr_n, color=color)
            normed_aucs[col_name].append(roc_auc_n)
        else:
            axes[0, thresh_idx].text(
                0.5, 0.5,
                f"Only one class\n({col_name})",
                ha="center", va="center",
                fontsize=title_fontsize,
                color=color
            )
            normed_aucs[col_name].append(np.nan)
    for row_idx in range(1):
        axes[row_idx, thresh_idx].plot([0, 1], [0, 1], "k--", linewidth=0.7)
        axes[row_idx, thresh_idx].set_title(f"log(Activity) > {t}", fontsize=title_fontsize)
        axes[row_idx, thresh_idx].set_xlabel("FPR", fontsize=label_fontsize)
        if thresh_idx == 0:
            axes[row_idx, thresh_idx].set_ylabel("TPR", fontsize=label_fontsize)
        axes[row_idx, thresh_idx].tick_params(axis='both', labelsize=tick_fontsize)
        axes[row_idx, thresh_idx].set_yticks([0, 0.5, 1])
        axes[row_idx, thresh_idx].legend(loc="lower right", fontsize=legend_fontsize)

fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

plt.figure(figsize=(3, 3))
data = [raw_aucs[col_name] for col_name in normed_fitness.columns]
box = plt.boxplot(data, labels=normed_fitness.columns, patch_artist=True)
for patch, col_name in zip(box['boxes'], normed_fitness.columns):
    color = color_map.get(col_name, None)
    if color is not None:
        patch.set_facecolor(color)
    else:
        patch.set_facecolor('lightgrey')
plt.xlabel("Model Name", fontsize=label_fontsize)
plt.ylabel("ROC AUC", fontsize=label_fontsize)
plt.xticks(fontsize=tick_fontsize)
plt.yticks([0, 0.5, 1], fontsize=tick_fontsize)
plt.tight_layout()
plt.show()
output_dir = "./figures/figure_1"
os.makedirs(output_dir, exist_ok=True)
fig.savefig(os.path.join(output_dir, "nmt_roc_curves_fixed.svg"), bbox_inches='tight')


In [ ]:
dataset_to_use = "gfp"

datasets_path = "./notebooks/%s/fitness_results/" % dataset_to_use
df = pd.read_csv(datasets[dataset_to_use])

normed_fitness_path = datasets_path + "normed_fitness_all.csv"
normed_fitness = pd.read_csv(normed_fitness_path)

fitness_path = datasets_path + "fitness_all.csv"
fitness = pd.read_csv(fitness_path)

In [ ]:
gfp_labels = (df["inactive"] == False).astype(int)



nrows, ncols = 1, 2
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 6), squeeze=False)

# axes[0,0]: normed fitness, axes[0,1]: raw fitness
for col_idx, col_name in enumerate(fitness.columns):
    valid = (~fitness[col_name].isna()) & (~normed_fitness[col_name].isna())
    y_true = gfp_labels.loc[valid]
    fit_col = fitness.loc[valid, col_name]
    normed_fit_col = normed_fitness.loc[valid, col_name]
    color = color_map.get(col_name, None)

    # Plot for normed fitness
    if np.unique(y_true).size == 2:
        fpr_n, tpr_n, _ = roc_curve(y_true, normed_fit_col)
        roc_auc_n = auc(fpr_n, tpr_n)
        label_n = f"{col_name} (AUC={roc_auc_n:.2f})"
        axes[0, 0].plot(fpr_n, tpr_n, label=label_n, color=color)
    else:
        axes[0, 0].text(0.5, 0.5, f"Only one class\n({col_name})", ha="center", va="center", fontsize=10, color=color)

    # Plot for raw fitness
    if np.unique(y_true).size == 2:
        fpr, tpr, _ = roc_curve(y_true, fit_col)
        roc_auc = auc(fpr, tpr)
        label = f"{col_name} (AUC={roc_auc:.2f})"
        axes[0, 1].plot(fpr, tpr, label=label, color=color)
    else:
        axes[0, 1].text(0.5, 0.5, f"Only one class\n({col_name})", ha="center", va="center", fontsize=10, color=color)

# Formatting and random line
for ax, title in zip(axes[0],
                     ["Normed Fitness ROC (gfp_labels)", "Raw Fitness ROC (gfp_labels)"]):
    ax.plot([0, 1], [0, 1], "k--", linewidth=0.7)
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.set_title(title)
    ax.legend(loc="lower right", fontsize=9)

fig.suptitle("ROC Curves: all models, GFP labels\nLeft: Normed Fitness, Right: Raw Fitness", fontsize=18, y=1.05)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


In [ ]:

mut_range = [2, 3, 4, 5, 6, 7, 8]
nrows, ncols = 1, len(mut_range)
fig, axes = plt.subplots(nrows, ncols, figsize=(8.25, 1.65), squeeze=False)

all_aucs = []
for idx, mut_val in enumerate(mut_range):
    mask = (df["num_muts"] == mut_val)
    y_true_mut = (df.loc[mask, "inactive"] == False).astype(int)
    fit_mut = fitness.loc[mask]
    normed_fit_mut = normed_fitness.loc[mask]
    col_aucs = []

    for col_idx, col_name in enumerate(fitness.columns):
        valid = (~fit_mut[col_name].isna()) & (~normed_fit_mut[col_name].isna())
        y_valid = y_true_mut.loc[valid]
        fit_col = fit_mut.loc[valid, col_name]
        normed_fit_col = normed_fit_mut.loc[valid, col_name]
        color = color_map.get(col_name, None)

        if np.unique(y_valid).size == 2:
            fpr, tpr, _ = roc_curve(y_valid, fit_col)
            roc_auc = auc(fpr, tpr)
            col_aucs.append(roc_auc)
            axes[0, idx].plot(fpr, tpr, color=color)  # removed label
        else:
            axes[0, idx].text(
                0.5,
                0.5,
                f"Only one class\n({col_name})",
                ha="center",
                va="center",
                fontsize=title_fontsize,
                color=color
            )
    all_aucs.append(col_aucs)    
    axes[0, idx].plot([0, 1], [0, 1], "k--", linewidth=0.7)
    axes[0, idx].set_xlabel("FPR", fontsize=label_fontsize)
    if idx == 0:
        axes[0, idx].set_ylabel("TPR", fontsize=label_fontsize)
    else:
        axes[0, idx].set_ylabel("")
    axes[0, idx].set_title(f"(#{mut_val} muts)", fontsize=title_fontsize)
    axes[0, idx].tick_params(axis='both', labelsize=tick_fontsize)
    # Removed legend
    
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()
output_dir = "./figures/figure_1"
os.makedirs(output_dir, exist_ok=True)
fig.savefig(os.path.join(output_dir, "gfp_roc_curves.svg"), bbox_inches='tight')


In [ ]:
labels = {0: 'Inactive', 1: 'Active'}
colors = {0: 'skyblue', 1: 'salmon'}

import seaborn as sns

y_true = (df["inactive"] == False).astype(int)

columns = normed_fitness.columns
fig, axes = plt.subplots(1, len(columns), figsize=(8.25, 1.65), squeeze=False)

for i, col in enumerate(columns):
    ax = axes[0, i]
    data_plot = pd.DataFrame({
        'Fitness': normed_fitness[col],
        'Label': y_true.map(labels)
    })
    sns.violinplot(
        data=data_plot,
        x='Label',
        y='Fitness',
        hue='Label',
        ax=ax,
        palette=[colors[0], colors[1]],
        cut=0,
        linewidth=1,
        inner="box",  # Changed from "median" to valid value "box"
        legend=False
    )
    ax.set_xticklabels([labels[0], labels[1]], rotation=45, fontsize=tick_fontsize)
    ax.tick_params(axis='both', labelsize=tick_fontsize)
    if i == 0:
        ax.set_ylabel("Fitness", fontsize=label_fontsize)
    else:
        ax.set_ylabel("", fontsize=label_fontsize)
    ax.set_xlabel("", fontsize=label_fontsize)
    ax.set_title(col, fontsize=title_fontsize)

# Commented out old violinplot code
#     data = [
#         nf_np[label_masks[val], col_idx][~np.isnan(nf_np[label_masks[val], col_idx])]
#         for val in [0, 1]
#     ]
#     vp = ax.violinplot(
#         data,
#         positions=[1, 2],
#         showmedians=True,
#         widths=0.8
#     )
#     for j, b in enumerate(vp['bodies']):
#         b.set_facecolor(colors[j])
#         b.set_edgecolor("k")
#         b.set_alpha(0.7)
#     if 'cmedians' in vp:
#         vp['cmedians'].set_color('black')
#         vp['cmedians'].set_linewidth(1.5)
#     ax.set_xticks([1, 2])
#     ax.set_xticklabels([labels[0], labels[1]], rotation=45, fontsize=tick_fontsize)
#     ax.tick_params(axis='both', labelsize=tick_fontsize)
#     if i == 0:
#         ax.set_ylabel("Fitness", fontsize=label_fontsize)
#     else:
#         ax.set_ylabel("", fontsize=label_fontsize)

fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
output_dir = "./figures/figure_1"
os.makedirs(output_dir, exist_ok=True)
fig.savefig(os.path.join(output_dir, "gfp_raw_activity_.svg"), bbox_inches='tight')
